# IN6227 Assignment 1 — Variant 1
## Part 4: Model Training and Hyperparameter Tuning

In [1]:
import time
import joblib
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
RANDOM_STATE = 42
TARGET = "label"

train = pd.read_csv("train_clean.csv")
X_train = train.drop(columns=TARGET)
y_train = train[TARGET]
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]
print("X_train:", X_train.shape, "| positive rate:", round(y_train.mean(), 3))

X_train: (31109, 15) | positive rate: 0.24


### 4.1 Preprocessing (same as Part 3)

In [3]:
def make_preprocessor(scale, drop=()):
    num = [c for c in num_cols if c not in drop]
    cat = [c for c in cat_cols if c not in drop]
    num_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("scale", StandardScaler()))
    cat_steps = [("impute", SimpleImputer(strategy="most_frequent")),
                 ("onehot", OneHotEncoder(handle_unknown="ignore"))]
    return ColumnTransformer([("num", Pipeline(num_steps), num),
                              ("cat", Pipeline(cat_steps), cat)])

LR_DROP = ["composite_rank", "index_weight", "region", "geological_era"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
SCORING = {"f1": "f1", "roc_auc": "roc_auc"}

### 4.2 Helper functions

In [5]:
def summarise(search, top=5):
    res = pd.DataFrame(search.cv_results_)
    cols = ["params", "mean_test_f1", "std_test_f1", "mean_test_roc_auc", "mean_train_f1", "mean_fit_time"]
    out = res.sort_values("rank_test_f1")[cols].head(top).copy()
    out.columns = ["params", "CV F1", "CV F1 std", "CV AUC", "train F1", "fit time (s)"]
    return out.round(4)

def edge_check(search, grid):
    for name, values in grid.items():
        if len(values) < 3:
            continue
        best = search.best_params_[name]
        pos = values.index(best)
        if best is None:
            where = "no limit = the most extreme option, nothing further to try"
        elif pos in (0, len(values) - 1):
            where = "EDGE of grid -> consider extending"
        else:
            where = "inside grid (OK)"
        print(f"  {name}: best = {best}  ({where})")

### 4.3 Logistic Regression

In [16]:
lr_pipe = Pipeline([
    ("pre", make_preprocessor(scale=True, drop=LR_DROP)),
    ("model", LogisticRegression(max_iter=1000, tol=1e-4)),
])
lr_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__class_weight": [None, "balanced"],
}
lr_search = GridSearchCV(lr_pipe, lr_grid, cv=cv, scoring=SCORING, refit="f1",
                         return_train_score=True, n_jobs=1)
t0 = time.time()
lr_search.fit(X_train, y_train)
print(f"LR grid search: {len(lr_search.cv_results_['params'])} combinations x 5 folds, "
      f"{time.time() - t0:.1f} s")
print("Best params:", lr_search.best_params_)
edge_check(lr_search, lr_grid)
summarise(lr_search)

LR grid search: 10 combinations x 5 folds, 29.6 s
Best params: {'model__C': 10, 'model__class_weight': 'balanced'}
  model__C: best = 10  (inside grid (OK))


,params,CV F1,CV F1 std,CV AUC,train F1,fit time (s)
7,"{'model__C': 10, 'model__class_weight': 'balanced'}",0.6596,0.0032,0.8902,0.6615,0.3641
3,"{'model__C': 0.1, 'model__class_weight': 'balanced'}",0.6595,0.0040,0.8899,0.6601,0.2983
9,"{'model__C': 100, 'model__class_weight': 'balanced'}",0.6594,0.0029,0.8903,0.6615,0.4138
5,"{'model__C': 1, 'model__class_weight': 'balanced'}",0.6594,0.0034,0.8901,0.6607,0.3603
1,"{'model__C': 0.01, 'model__class_weight': 'balanced'}",0.6563,0.0029,0.8876,0.6570,0.2368


In [18]:
best_lr = lr_search.best_estimator_.named_steps["model"]
print("Iterations used:", best_lr.n_iter_[0], "of max_iter =", best_lr.max_iter)

Iterations used: 106 of max_iter = 1000


### 4.4 Random Forest

In [21]:
# How deep do trees grow with no limit?
probe = Pipeline([("pre", make_preprocessor(scale=False)),
                  ("model", RandomForestClassifier(n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1))])
probe.fit(X_train, y_train)
depths = [t.get_depth() for t in probe.named_steps["model"].estimators_]
print("Unconstrained tree depth -> min:", min(depths), "| median:", int(np.median(depths)), "| max:", max(depths))

Unconstrained tree depth -> min: 36 | median: 46 | max: 55


In [24]:
rf_pipe = Pipeline([
    ("pre", make_preprocessor(scale=False)),
    ("model", RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)),
])
rf_grid = {
    "model__max_depth": [10, 20, None],
    "model__min_samples_leaf": [1, 5, 20],
    "model__class_weight": [None, "balanced"],
}
rf_search = GridSearchCV(rf_pipe, rf_grid, cv=cv, scoring=SCORING, refit="f1",
                         return_train_score=True, n_jobs=1)
t0 = time.time()
rf_search.fit(X_train, y_train)
print(f"RF grid search: {len(rf_search.cv_results_['params'])} combinations x 5 folds, "
      f"{time.time() - t0:.1f} s")

print("Best params:", rf_search.best_params_)
edge_check(rf_search, rf_grid)
summarise(rf_search)

RF grid search: 18 combinations x 5 folds, 231.6 s
Best params: {'model__class_weight': 'balanced', 'model__max_depth': None, 'model__min_samples_leaf': 5}
  model__max_depth: best = None  (no limit = the most extreme option, nothing further to try)
  model__min_samples_leaf: best = 5  (inside grid (OK))


,params,CV F1,CV F1 std,CV AUC,train F1,fit time (s)
16,"{'model__class_weight': 'balanced', 'model__max_depth': ...",0.6680,0.0057,0.8905,0.7621,2.3018
13,"{'model__class_weight': 'balanced', 'model__max_depth': ...",0.6659,0.0048,0.8908,0.7403,2.1483
17,"{'model__class_weight': 'balanced', 'model__max_depth': ...",0.6612,0.0039,0.8902,0.6876,1.6099
14,"{'model__class_weight': 'balanced', 'model__max_depth': ...",0.6609,0.0043,0.8901,0.6865,1.5634
12,"{'model__class_weight': 'balanced', 'model__max_depth': ...",0.6587,0.0088,0.8863,0.8678,2.7603


### 4.5 Effect of `class_weight`

In [26]:
rows = []
for name, search in [("LR", lr_search), ("RF", rf_search)]:
    res = pd.DataFrame(search.cv_results_)
    res["cw"] = [str(p["model__class_weight"]) for p in search.cv_results_["params"]]
    for cw in ["None", "balanced"]:
        sub = res[res["cw"] == cw]
        best = sub.sort_values("mean_test_f1", ascending=False).iloc[0]
        rows.append([name, cw, best["mean_test_f1"], best["mean_test_roc_auc"]])
pd.DataFrame(rows, columns=["model", "class_weight", "best CV F1", "CV AUC"]).round(4)

,model,class_weight,best CV F1,CV AUC
0,LR,None,0.6244,0.8904
1,LR,balanced,0.6596,0.8902
2,RF,None,0.6237,0.8876
3,RF,balanced,0.6680,0.8905


### 4.6 Effect of the stopping criteria on RF (overfitting check)

In [28]:
res = pd.DataFrame(rf_search.cv_results_)
params = rf_search.cv_results_["params"]
res["cw"] = [str(p["model__class_weight"]) for p in params]
res["max_depth"] = [str(p["model__max_depth"]) for p in params]
res["min_samples_leaf"] = [p["model__min_samples_leaf"] for p in params]
sub = res[res["cw"] == str(rf_search.best_params_["model__class_weight"])].copy()
sub["gap"] = sub["mean_train_f1"] - sub["mean_test_f1"]
overfit_table = sub[["max_depth", "min_samples_leaf", "mean_train_f1", "mean_test_f1", "gap"]].rename(
    columns={"mean_train_f1": "train F1", "mean_test_f1": "CV F1"}).round(4)
overfit_table

,max_depth,min_samples_leaf,train F1,CV F1,gap
9,10,1,0.6891,0.6538,0.0353
10,10,5,0.6774,0.6518,0.0256
11,10,20,0.6677,0.6516,0.0161
12,20,1,0.8678,0.6587,0.2091
13,20,5,0.7403,0.6659,0.0744
14,20,20,0.6865,0.6609,0.0256
15,None,1,0.9999,0.6043,0.3956
16,None,5,0.7621,0.6680,0.0942
17,None,20,0.6876,0.6612,0.0264


### 4.7 Final models: refit on the full training set and save

In [30]:
from sklearn.base import clone
from sklearn.model_selection import cross_validate

final_lr = lr_search.best_estimator_
final_rf = clone(rf_search.best_estimator_).set_params(model__n_estimators=300)

# Confirm the 300-tree RF with the same 5 folds
rf300_cv = cross_validate(final_rf, X_train, y_train, cv=cv, scoring=SCORING, return_train_score=True)
print(f"RF with 300 trees -> CV F1 {rf300_cv['test_f1'].mean():.4f} ± {rf300_cv['test_f1'].std():.4f}, "
      f"CV AUC {rf300_cv['test_roc_auc'].mean():.4f}  "
      f"(search with 100 trees: {rf_search.best_score_:.4f})")

RF with 300 trees -> CV F1 0.6685 ± 0.0038, CV AUC 0.8913  (search with 100 trees: 0.6680)


In [31]:
lr_best = pd.DataFrame(lr_search.cv_results_).iloc[lr_search.best_index_]
rows = []
for name, model, params, cv_f1, cv_std, cv_auc, tr_f1 in [
    ("Logistic Regression", final_lr, lr_search.best_params_,
     lr_best["mean_test_f1"], lr_best["std_test_f1"], lr_best["mean_test_roc_auc"], lr_best["mean_train_f1"]),
    ("Random Forest (300 trees)", final_rf, rf_search.best_params_,
     rf300_cv["test_f1"].mean(), rf300_cv["test_f1"].std(), rf300_cv["test_roc_auc"].mean(),
     rf300_cv["train_f1"].mean()),
]:
    t0 = time.time()
    model.fit(X_train, y_train)
    rows.append([name, params, cv_f1, cv_std, cv_auc, tr_f1, time.time() - t0])

summary = pd.DataFrame(rows, columns=["model", "best params", "CV F1", "CV F1 std",
                                      "CV AUC", "train F1", "fit time (s)"]).round(4)
joblib.dump(final_lr, "lr_model.joblib")
joblib.dump(final_rf, "rf_model.joblib")
print("Saved lr_model.joblib and rf_model.joblib")
summary

Saved lr_model.joblib and rf_model.joblib


,model,best params,CV F1,CV F1 std,CV AUC,train F1,fit time (s)
0,Logistic Regression,"{'model__C': 10, 'model__class_weight': 'balanced'}",0.6596,0.0032,0.8902,0.6615,0.5183
1,Random Forest (300 trees),"{'model__class_weight': 'balanced', 'model__max_depth': ...",0.6685,0.0038,0.8913,0.7644,8.2976
